Importación de modulos necesarios

In [1]:
import cv2  
import math 

from ultralytics import YOLO

**TAREA 1**

Carga y configuración de los modelos.

In [2]:
source_video = "C0142.mp4"
output_video = "output.mp4"

# Carga de los modelos: preentrenado y detección de matrículas
general_model = YOLO("yolo11n.pt")
plate_model = YOLO("./runs/detect/yolo_runs/plates_detection/weights/best.pt")

# Definición de las clases de interés
PERSON = 0
VEHICLES = {1, 2, 3, 5} # bicicleta, coche, moto, guagua
PLATE_VEHICLES = {2, 3, 5}   # solo los que llevan matrícula
CLASSES = [PERSON, *VEHICLES]

# Se configuran los parámetros de detección y recorte de vehículos 
conf_coco = 0.4
conf_plate = 0.3
padding = 0.05 # margen extra alrededor del vehículo (5%)

# Se definen los colores asociados a cada una de las detecciones
class_colors = {
    0: (0, 200, 0),      # persona
    1: (200, 0, 200),    # bicicleta
    2: (255, 120, 0),    # coche
    3: (0, 165, 255),    # moto
    5: (0, 200, 255),    # guagua
}
plate_color = (0, 0, 255)

Configuración para preparar el vídeo

In [3]:
vid = cv2.VideoCapture(source_video)
fps = vid.get(cv2.CAP_PROP_FPS)
w = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

Se define la función auxiliar draw, para dibujar las cajas de las detecciones.

In [4]:
def draw(frame, box, label, color):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(frame, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
    cv2.putText(frame, label, (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

In [5]:
while True:
    ret, frame = vid.read()
    if not ret:
        break

    r = general_model.track(frame, persist=True, classes=CLASSES,
                   conf=conf_coco, verbose=True)[0]

    crops, offsets = [], []
    for box, cls, conf, tid in zip(
        r.boxes.xyxy.cpu().numpy(),
        r.boxes.cls.cpu().numpy().astype(int),
        r.boxes.conf.cpu().numpy(),
        r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None
        else [None] * len(r.boxes),
    ):
        x1, y1, x2, y2 = box
        name = general_model.names[cls]
        tag = f"{name} #{tid}" if tid is not None else name
        is_vehicle = cls in VEHICLES
        draw(frame, box, f"{tag} {conf:.2f}", class_colors.get(cls, (255, 255, 255)))

        if cls in PLATE_VEHICLES:
            bw, bh = x2 - x1, y2 - y1
            cx1 = max(0, int(x1 - bw * padding)); cy1 = max(0, int(y1 - bh * padding))
            cx2 = min(w, int(x2 + bw * padding)); cy2 = min(h, int(y2 + bh * padding))
            crop = frame[cy1:cy2, cx1:cx2]
            if crop.size:
                crops.append(crop)
                offsets.append((cx1, cy1))

    if crops:
        for res, (ox, oy) in zip(plate_model.predict(crops, conf=conf_plate, verbose=False), offsets):
            for pbox, pconf in zip(res.boxes.xyxy.cpu().numpy(),
                                   res.boxes.conf.cpu().numpy()):
                px1, py1, px2, py2 = pbox
                draw(frame, (px1 + ox, py1 + oy, px2 + ox, py2 + oy),
                     f"plate {pconf:.2f}", plate_color)
    out.write(frame)

vid.release()
out.release()
print("Vídeo con las detecciones generado:", output_video)


0: 384x640 1 car, 1 bus, 13.1ms
Speed: 206.5ms preprocess, 13.1ms inference, 248.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 9.2ms
Speed: 1.5ms preprocess, 9.2ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 8.9ms
Speed: 1.3ms preprocess, 8.9ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 8.8ms
Speed: 1.7ms preprocess, 8.8ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 8.5ms
Speed: 1.8ms preprocess, 8.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 9.4ms
Speed: 1.2ms preprocess, 9.4ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 9.6ms
Speed: 1.2ms preprocess, 9.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 bus, 10.5ms
Speed: 1.3ms preprocess, 10.5ms inference, 1.8ms postprocess per 

**TAREA 2**